In [0]:
# --- KONFIGURACJA ---
acc_name = "medallionazure"
acc_key = dbutils.secrets.get(scope="azure-storage", key="storagekeybdg")
spark.conf.set(f"fs.azure.account.key.{acc_name}.blob.core.windows.net", acc_key)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# Event Hubs connection
eh_conn_str = dbutils.secrets.get(scope="azure-storage", key="eventhubs-connection-string")
eh_name = "taxi-rides"

# Pełny connection string z EventHub name
eh_conf = {
    "eventhubs.connectionString": sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(
        f"{eh_conn_str};EntityPath={eh_name}"
    )
}

print("✅ Konfiguracja załadowana")

In [0]:
eh_conn_str = dbutils.secrets.get(scope="azure-storage", key="eventhubs-connection-string")
print(f"Długość: {len(eh_conn_str)}")
print(f"Zaczyna się od 'Endpoint': {eh_conn_str.startswith('Endpoint')}")
print(f"Zawiera EntityPath: {'EntityPath' in eh_conn_str}")

In [0]:
from pyspark.sql.functions import to_json, struct

# Czytaj z Silver (czyste dane, spójny schemat)
silver_path = f"wasbs://data@{acc_name}.blob.core.windows.net/silver/yellow_taxi"
silver_df = spark.read.parquet(silver_path).limit(5000)

print(f"📦 Wczytano {silver_df.count()} rekordów z Silver")

# Event Hubs wymaga kolumny "body"
producer_df = silver_df.select(
    to_json(struct("*")).alias("body")
)

# Wyślij do Event Hubs
producer_df.write \
    .format("eventhubs") \
    .options(**eh_conf) \
    .save()

print("✅ Dane wysłane do Event Hubs!")